# ISL low-data study — one notebook, four team members

Settings: **Accelerator = GPU T4 x2**, **Internet = on**, **Persistence: Files** (optional).
Run with *Save Version → Save & Run All (Commit)* so it runs in the background for up to 12 h.

Set `STAGE` and `MEMBER` in the next cell. Every team member uses the same grid file and
`MEMBERS = 4`; the work is split by **K = training videos per word** (see `islr/lowdata/sweep.py`).

| STAGE | what it does | inputs to attach |
|---|---|---|
| `extract` | MediaPipe landmarks for shard `MEMBER` of a corpus → `/kaggle/working/stores/...` | nothing (downloads) or zips dataset |
| `merge` | one member merges the 4 shard outputs into one store | the 4 extract outputs |
| `sweep` | this member's share of the grid on both T4s, resumable | stores dataset (+ this notebook's previous output to resume) |
| `report` | tables, plots, paired tests over all members | the 4 sweep outputs |

Nothing here uploads anything. Stores and results stay in this notebook's output until
you choose to turn them into a (private) dataset.

In [ ]:
STAGE = "sweep"          # extract | merge | sweep | report
MEMBER = 0               # 0..3 — each team member uses their own number
MEMBERS = 4
GRID = "islr/lowdata/configs/scarce_legacy8.json"
TIME_BUDGET_H = 11.3     # Kaggle stops at 12 h; runs checkpoint before this

# where the code comes from: a private Kaggle dataset with the repo, or a git branch
CODE_DATASET = "/kaggle/input/isl-lowdata-slr-code"   # used if it exists
REPO_URL = "https://github.com/Vidit-01/isl-lowdata-slr.git"
BRANCH = "main"

# landmark stores (private Kaggle dataset made from the extract/merge outputs)
STORES = "/kaggle/input/isl-lowdata-stores"
# extract stage
EXTRACT_SOURCE = "include"   # include | isl40 | isl_dictionary | cislr

In [ ]:
import glob, os, shutil, subprocess, sys
WORK = "/kaggle/working"
CODE = "/kaggle/tmp/code" if os.path.isdir("/kaggle/tmp") else "/tmp/code"
if os.path.isdir(CODE_DATASET):
    shutil.copytree(CODE_DATASET, CODE, dirs_exist_ok=True)
elif not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)
os.environ["STORES"] = STORES
os.environ["PYTHONUNBUFFERED"] = "1"
try:  # Kaggle Secrets (Add-ons → Secrets): HF_TOKEN for gated/private Hugging Face data
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass

def sh(*args, env=None):
    print("$", " ".join(map(str, args)), flush=True)
    r = subprocess.run([str(a) for a in args], env=env)
    print("exit code", r.returncode)
    return r.returncode

import torch
print(torch.__version__, torch.cuda.device_count(), "GPU(s)",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Stage `extract` (CPU-bound; GPU not needed)

MediaPipe is installed into its own folder and only the extraction subprocess sees it, so the
notebook's torch/numpy stay untouched. Each member extracts shard `MEMBER` of `MEMBERS`;
the store in `/kaggle/working/stores` is this notebook's output. If the time budget runs out,
attach this output as an input, and the next run copies it back and continues.

In [ ]:
if STAGE == "extract":
    MP = "/tmp/mp"
    if not os.path.isdir(MP):
        sh(sys.executable, "-m", "pip", "install", "-q", "--target", MP, "mediapipe==0.10.21", "opencv-python-headless")
    env = dict(os.environ, PYTHONPATH=MP, HOLISTIC_TASK_PATH="/tmp/holistic_landmarker.task")
    store = f"{WORK}/stores/{EXTRACT_SOURCE}_shard{MEMBER}"
    # continue a previous session: copy its store back into the working dir
    for prev in glob.glob(f"/kaggle/input/*/stores/{EXTRACT_SOURCE}_shard{MEMBER}"):
        shutil.copytree(prev, store, dirs_exist_ok=True)
    args = [sys.executable, "lowdata.py", "sources", EXTRACT_SOURCE, "--store", store,
            "--shard", f"{MEMBER}/{MEMBERS}", "--workers", str(os.cpu_count()), "--work", "/tmp/dl"]
    if EXTRACT_SOURCE in ("include", "isl40"):
        args += ["--time-budget-h", str(TIME_BUDGET_H - 0.5)]
    sh(*args, env=env)

In [ ]:
if STAGE == "merge":
    for src in ("include", "isl40", "isl_dictionary", "cislr"):
        shards = sorted(glob.glob(f"/kaggle/input/*/stores/{src}_shard*"))
        if shards:
            sh(sys.executable, "lowdata.py", "sources", "merge", "--from", *shards, "--store", f"{WORK}/stores/{src}")
    # then: Output → New Dataset (private) → name it isl-lowdata-stores

## Stage `sweep`

1. quick GPU smoke test (2–3 min): every path on the T4, fp16 on;
2. `inspect`: store statistics and whether every K is feasible for every seed;
3. this member's share of the grid: one job queue per T4 (see `sweep.py` for why not DDP
   for these small jobs); results in `/kaggle/working/sweeps`.

**Resuming after 12 h:** add this notebook's previous version as an input (Add Input →
Your Work → this notebook). Its finished runs are copied and skipped, and interrupted runs continue
from their checkpoint.

In [ ]:
if STAGE == "sweep":
    sh(sys.executable, "lowdata.py", "smoke", "--fast", "--device", "cuda", "--skip", "ddp", "sweep")
    stores = [os.path.expandvars(s) for s in __import__("json").load(open(GRID))["stores"]]
    sh(sys.executable, "lowdata.py", "inspect", "--stores", *stores, "--grid", GRID, "--dest", f"{WORK}/inspect")
    sh(sys.executable, "lowdata.py", "sweep", "--grid", GRID, "--plan", "--members", str(MEMBERS))

In [ ]:
if STAGE == "sweep":
    resume = sorted(glob.glob("/kaggle/input/*/sweeps"))
    rc = sh(sys.executable, "lowdata.py", "sweep", "--grid", GRID, "--member", str(MEMBER), "--members", str(MEMBERS),
            "--out", f"{WORK}/sweeps", "--cache-dir", "/tmp/bank", "--time-budget-h", str(TIME_BUDGET_H),
            *(["--resume-from", *resume] if resume else []))
    print("ALL DONE" if rc == 0 else "NOT FINISHED: save this version, attach its output, and run again")
    sh(sys.executable, "lowdata.py", "report", "--out", f"{WORK}/sweeps", "--dest", f"{WORK}/sweeps/_report")

## Stage `report`

Attach the final sweep outputs of all four members as inputs.

In [ ]:
if STAGE == "report":
    outs = sorted(glob.glob("/kaggle/input/*/sweeps"))
    print(outs)
    sh(sys.executable, "lowdata.py", "report", "--out", *outs, "--dest", f"{WORK}/report")
    from IPython.display import Markdown, display
    display(Markdown(open(f"{WORK}/report/summary.md").read()))